# View All Subjects' Training Data

This notebook loads the latest subject-specific training datasets, shows each subject’s selected four activities and label mapping, displays all window metadata, and previews a complete 128-timestep sensor window for every subject. A final cell exports every training timestep and feature value to a separate long-format CSV per subject.

Run from the workspace root or from inside `DATA_PREP`. The notebook reads the latest selection folder under `DATA_PREP/processed_datasets_from_activity_config/`. Values in `X_train.npy` are standardized sensor values. Only original `_1_extracted` recordings are used for training.

S4 is included with its four available training activities. It has no `_2_extracted` folders, so its test array is intentionally empty; `_extracted_r` training variants are not treated as test data.

In [12]:
from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

current_directory = Path.cwd().resolve()
OUTPUT_PARENT_NAME = 'processed_datasets_from_activity_config'
DATA_PREP_DIR = None
for candidate in (current_directory, *current_directory.parents):
    if candidate.name.lower() == 'data_prep' and (candidate / OUTPUT_PARENT_NAME).is_dir():
        DATA_PREP_DIR = candidate
        break
    nested_data_prep = candidate / 'DATA_PREP'
    if (nested_data_prep / OUTPUT_PARENT_NAME).is_dir():
        DATA_PREP_DIR = nested_data_prep
        break
if DATA_PREP_DIR is None:
    raise FileNotFoundError(f'Could not locate DATA_PREP/{OUTPUT_PARENT_NAME} above {current_directory}')

OUTPUT_PARENT = DATA_PREP_DIR / OUTPUT_PARENT_NAME
prepared_roots = sorted(
    (path for path in OUTPUT_PARENT.glob('selection_*') if (path / 'subject_preparation_summary.csv').is_file()),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not prepared_roots:
    raise FileNotFoundError(f'No completed selection_* dataset found under {OUTPUT_PARENT}')
PREPARED_ROOT = prepared_roots[0]
SUBJECT_SUMMARY = pd.read_csv(PREPARED_ROOT / 'subject_preparation_summary.csv')
CLASS_MAPPINGS = json.loads((PREPARED_ROOT / 'class_mappings_by_subject.json').read_text(encoding='utf-8'))
SUBJECT_TRAINING = {}

print(f'Prepared data root: {PREPARED_ROOT}')
print('Subject preparation status:')
display(SUBJECT_SUMMARY)

for _, row in SUBJECT_SUMMARY.iterrows():
    if not str(row['status']).startswith('prepared'):
        continue
    subject = row['subject']
    subject_dir = PREPARED_ROOT / subject
    X = np.load(subject_dir / 'X_train.npy')
    y = np.load(subject_dir / 'y_train.npy')
    metadata = pd.read_csv(subject_dir / 'train_metadata.csv')
    feature_names = json.loads((subject_dir / 'feature_names.json').read_text(encoding='utf-8'))
    class_mapping = CLASS_MAPPINGS[subject]
    class_names = {label: activity for activity, label in class_mapping.items()}

    if not metadata['original_folder'].str.endswith('_1_extracted').all():
        raise ValueError(f'{subject} training metadata contains a non-training source folder.')
    if len(X) != len(y) or len(X) != len(metadata):
        raise ValueError(f'{subject} X_train, y_train, and metadata lengths do not match.')
    if set(np.unique(y).tolist()) != set(class_mapping.values()):
        raise ValueError(f'{subject} y_train labels do not match its subject-specific class mapping.')

    SUBJECT_TRAINING[subject] = {
        'X': X,
        'y': y,
        'metadata': metadata,
        'feature_names': feature_names,
        'class_mapping': class_mapping,
        'class_names': class_names,
        'directory': subject_dir,
        'status': row['status'],
    }

summary_rows = []
for subject, data in SUBJECT_TRAINING.items():
    counts = Counter(data['y'].tolist())
    summary_rows.append({
        'subject': subject,
        'status': data['status'],
        'class_mapping': data['class_mapping'],
        'X_train_shape': str(data['X'].shape),
        'y_train_shape': str(data['y'].shape),
        'feature_count': len(data['feature_names']),
        'training_windows': len(data['X']),
        'label_counts': {data['class_names'][label]: counts[label] for label in sorted(data['class_names'])},
        'source_folders': ', '.join(sorted(data['metadata']['original_folder'].unique())),
    })
print('Training dataset overview, including S4 training-only data:')
display(pd.DataFrame(summary_rows))

Prepared data root: D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\DATA_PREP\processed_datasets_from_activity_config\selection_14b5013a3e02
Subject preparation status:


,subject,status,selected_activities,missing_test_activities,train_windows,test_windows,output_dir
0,S1,prepared,"Bending,Bring,CleanFloor,CleanSurface",NaN,38,37,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
1,S2,prepared,"CloseBigBox,CloseDoor,CloseDrawer,CloseLidByRo...",NaN,25,25,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
2,S3,prepared,"closeotherlid,closesmallbox,drink,dryoffhand",NaN,28,29,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
3,S4,prepared_training_only_no_test_data,"DryOffHandByShake,eatsmallthing,Gargle,GetUp","DryOffHandByShake,Gargle,GetUp,eatsmallthing",36,0,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
4,S5,prepared,"Hang,LieDown,Lying,OpenBag",NaN,37,40,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
5,S6,prepared,"OpenBigBox,OpenDoor,OpenDrawer,OpenLidByRotate",NaN,31,31,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
6,S7,prepared,"Dryoffhands,Openotherlid,Opensmallbox,Plugin",NaN,30,31,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
7,S8,prepared,"PressByGrasp,Pressfromtop,PressSwitch,PutFromB...",NaN,29,27,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...


Training dataset overview, including S4 training-only data:


,subject,status,class_mapping,X_train_shape,y_train_shape,feature_count,training_windows,label_counts,source_folders
0,S1,prepared,"{'Bending': 0, 'Bring': 1, 'CleanFloor': 2, 'C...","(38, 128, 33)","(38,)",33,38,"{'Bending': 10, 'Bring': 10, 'CleanFloor': 11,...","S1_Bending_1_extracted, S1_Bring_1_extracted, ..."
1,S2,prepared,"{'CloseBigBox': 0, 'CloseDoor': 1, 'CloseDrawe...","(25, 128, 33)","(25,)",33,25,"{'CloseBigBox': 10, 'CloseDoor': 5, 'CloseDraw...","S2_CloseBigBox_1_extracted, S2_CloseDoor_1_ext..."
2,S3,prepared,"{'closeotherlid': 0, 'closesmallbox': 1, 'drin...","(28, 128, 33)","(28,)",33,28,"{'closeotherlid': 7, 'closesmallbox': 7, 'drin...","S3_closeotherlid_1_extracted, S3_closesmallbox..."
3,S4,prepared_training_only_no_test_data,"{'DryOffHandByShake': 0, 'eatsmallthing': 1, '...","(36, 128, 33)","(36,)",33,36,"{'DryOffHandByShake': 10, 'eatsmallthing': 7, ...","S4_DryOffHandByShake_1_extracted, S4_Gargle_1_..."
4,S5,prepared,"{'Hang': 0, 'LieDown': 1, 'Lying': 2, 'OpenBag...","(37, 128, 33)","(37,)",33,37,"{'Hang': 10, 'LieDown': 10, 'Lying': 7, 'OpenB...","S5_Hang_1_extracted, S5_LieDown_1_extracted, S..."
5,S6,prepared,"{'OpenBigBox': 0, 'OpenDoor': 1, 'OpenDrawer':...","(31, 128, 33)","(31,)",33,31,"{'OpenBigBox': 10, 'OpenDoor': 7, 'OpenDrawer'...","S6_OpenBigBox_1_extracted, S6_OpenDoor_1_extra..."
6,S7,prepared,"{'Dryoffhands': 0, 'Openotherlid': 1, 'Opensma...","(30, 128, 33)","(30,)",33,30,"{'Dryoffhands': 10, 'Openotherlid': 7, 'Opensm...","S7_Dryoffhands_1_extracted, S7_Openotherlid_1_..."
7,S8,prepared,"{'PressByGrasp': 0, 'Pressfromtop': 1, 'PressS...","(29, 128, 33)","(29,)",33,29,"{'PressByGrasp': 7, 'Pressfromtop': 7, 'PressS...","S8_PressByGrasp_1_extracted, S8_PressSwitch_1_..."


In [7]:
print('All training-window labels and source metadata, grouped by subject:')
for subject, data in SUBJECT_TRAINING.items():
    metadata_view = data['metadata'].copy()
    metadata_view.insert(0, 'sample_index', np.arange(len(metadata_view)))
    print(f'\n{subject}: {len(metadata_view)} training windows')
    display(metadata_view)


def show_training_window(subject, sample_index=0):
    """Display one complete 128-timestep x feature-count training window."""
    data = SUBJECT_TRAINING[subject]
    if not 0 <= sample_index < len(data['X']):
        raise IndexError(f'{subject} sample_index must be between 0 and {len(data["X"]) - 1}.')
    row = data['metadata'].iloc[sample_index]
    label = int(data['y'][sample_index])
    window = pd.DataFrame(data['X'][sample_index], columns=data['feature_names'])
    window.index.name = 'timestep (20 ms each)'
    print(f'{subject} X_train[{sample_index}]: shape={window.shape}; label={label} ({row["activity"]})')
    print(f'Source: {row["original_folder"]} / {row["event"]} / window {row["window_id"]}')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
        display(window)


print('\nFull first training window for each available subject:')
for subject in SUBJECT_TRAINING:
    show_training_window(subject, sample_index=0)

All training-window labels and source metadata, grouped by subject:

S1: 38 training windows


,sample_index,subject,activity,train_or_test,activity_instance,event,window_id,label,original_folder,split
0,0,S1,Bending,train,1,Event_0,0,0,S1_Bending_1_extracted,train
1,1,S1,Bending,train,1,Event_1,0,0,S1_Bending_1_extracted,train
2,2,S1,Bending,train,1,Event_2,0,0,S1_Bending_1_extracted,train
3,3,S1,Bending,train,1,Event_3,0,0,S1_Bending_1_extracted,train
4,4,S1,Bending,train,1,Event_4,0,0,S1_Bending_1_extracted,train
5,5,S1,Bending,train,1,Event_5,0,0,S1_Bending_1_extracted,train
6,6,S1,Bending,train,1,Event_6,0,0,S1_Bending_1_extracted,train
7,7,S1,Bending,train,1,Event_7,0,0,S1_Bending_1_extracted,train
8,8,S1,Bending,train,1,Event_8,0,0,S1_Bending_1_extracted,train
9,9,S1,Bending,train,1,Event_9,0,0,S1_Bending_1_extracted,train



S2: 25 training windows


,sample_index,subject,activity,train_or_test,activity_instance,event,window_id,label,original_folder,split
0,0,S2,CloseBigBox,train,1,Event_0,0,0,S2_CloseBigBox_1_extracted,train
1,1,S2,CloseBigBox,train,1,Event_1,0,0,S2_CloseBigBox_1_extracted,train
2,2,S2,CloseBigBox,train,1,Event_2,0,0,S2_CloseBigBox_1_extracted,train
3,3,S2,CloseBigBox,train,1,Event_3,0,0,S2_CloseBigBox_1_extracted,train
4,4,S2,CloseBigBox,train,1,Event_4,0,0,S2_CloseBigBox_1_extracted,train
5,5,S2,CloseBigBox,train,1,Event_5,0,0,S2_CloseBigBox_1_extracted,train
6,6,S2,CloseBigBox,train,1,Event_6,0,0,S2_CloseBigBox_1_extracted,train
7,7,S2,CloseBigBox,train,1,Event_7,0,0,S2_CloseBigBox_1_extracted,train
8,8,S2,CloseBigBox,train,1,Event_8,0,0,S2_CloseBigBox_1_extracted,train
9,9,S2,CloseBigBox,train,1,Event_9,0,0,S2_CloseBigBox_1_extracted,train



S3: 28 training windows


...

,android.sensor.accelerometer_value_1,android.sensor.accelerometer_value_2,android.sensor.accelerometer_value_3,android.sensor.gyroscope_value_1,android.sensor.gyroscope_value_2,android.sensor.gyroscope_value_3,JinsAccelerometer_value_1,JinsAccelerometer_value_2,JinsAccelerometer_value_3,JinsBlinkSpeed_value_1,JinsBlinkStrength_value_1,JinsEyeMovement_value_1,JinsEyeMovement_value_2,JinsEyeMovement_value_3,JinsEyeMovement_value_4,JinsGyroscope_value_1,JinsGyroscope_value_2,JinsGyroscope_value_3,PhoneAccelerometer_value_1,PhoneAccelerometer_value_2,PhoneAccelerometer_value_3,PhoneGravity_value_1,PhoneGravity_value_2,PhoneGravity_value_3,PhoneGyroscope_value_1,PhoneGyroscope_value_2,PhoneGyroscope_value_3,PhoneLinearAccelerometer_value_1,PhoneLinearAccelerometer_value_2,PhoneLinearAccelerometer_value_3,PhoneMagnetometer_value_1,PhoneMagnetometer_value_2,PhoneMagnetometer_value_3
timestep (20 ms each),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,-1.487892,1.177012,0.191513,0.940159,2.361619,-1.087862,-0.730492,-0.342205,0.126286,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.011260,0.343963,0.390597,0.796706,1.690345,1.779439,0.716480,1.571402,1.729540,0.148991,0.182824,-1.284208,2.405558,1.250982,0.899722,-0.248717,-1.268001,-1.710220
1,-1.470775,1.308904,0.145471,1.290554,2.572993,-1.530317,-1.138256,0.265910,0.126286,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.007857,0.472305,0.394408,0.496212,1.628748,1.594522,0.705971,1.570816,1.722314,0.267613,-0.654028,-0.877003,-0.015484,0.914837,0.165442,-0.217645,-1.272433,-1.699837
2,-1.400232,1.612589,0.081011,1.421130,2.692230,-2.143342,-1.002334,0.367262,0.126286,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.018067,0.515329,0.395316,0.752866,1.644362,1.255818,0.700288,1.570547,1.713883,0.437480,-1.644793,-0.497165,-1.553287,0.392020,-1.372045,-0.221094,-1.275389,-1.705427
3,-1.362367,1.837582,0.037135,1.382615,2.789788,-2.806384,-0.866413,0.468615,0.126286,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.028276,0.558353,0.396224,1.081141,1.636913,1.416763,0.692643,1.570104,1.705979,0.449554,-2.456508,-0.899231,1.333508,0.615076,-3.228840,-0.193472,-1.269483,-1.702640
4,-1.389339,2.033757,-0.001865,1.267069,2.813273,-3.409150,-0.730492,0.569968,0.126286,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.038485,0.601377,0.397131,1.044951,1.637341,1.722001,0.679309,1.569208,1.700058,0.249104,-1.456629,-1.255126,2.494730,0.539929,-1.283889,-0.184494,-1.256174,-1.695856
5,-1.468700,2.159000,-0.012699,1.168432,2.804240,-3.808001,-0.628551,0.645982,0.338648,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.054934,0.631275,0.397539,0.965686,1.523745,1.821621,0.669619,1.568550,1.693126,0.172612,0.623072,-0.835954,2.364646,0.306209,0.546507,-0.178270,-1.246566,-1.683076
6,-1.564659,2.197791,0.019260,1.102674,2.695843,-3.968311,-0.526610,0.721996,0.551010,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.071382,0.661173,0.397948,0.582613,1.481345,1.744573,0.665949,1.568314,1.684342,0.059898,2.542796,-0.296989,1.640707,-0.761495,0.997318,-0.177582,-1.259131,-1.677478
7,-1.568809,2.250992,0.062594,1.072614,2.565767,-3.900339,-0.424669,0.798011,0.763372,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.087831,0.691070,0.398356,0.124863,1.553971,1.569928,0.664198,1.568191,1.675111,-0.075361,3.487210,0.009254,-1.606709,-1.001783,0.258474,-0.169292,-1.265782,-1.672685
8,-1.473887,2.253208,0.088595,1.092341,2.498922,-3.632301,-0.322728,0.874025,0.975735,-0.168854,-0.167354,-0.185207,-0.086365,-0.065985,-0.105868,-0.104279,0.720968,0.398765,0.231400,1.631469,1.353412,0.664844,1.568222,1.667581,-0.180014,2.940008,0.217982,-3.571746,0.070518,-0.920054,-0.157539,-1.269476,-1.674285


S7 X_train[0]: shape=(128, 33); label=0 (Dryoffhands)
Source: S7_Dryoffhands_1_extracted / Event_0 / window 0


,android.sensor.accelerometer_value_1,android.sensor.accelerometer_value_2,android.sensor.accelerometer_value_3,android.sensor.gyroscope_value_1,android.sensor.gyroscope_value_2,android.sensor.gyroscope_value_3,JinsAccelerometer_value_1,JinsAccelerometer_value_2,JinsAccelerometer_value_3,JinsBlinkSpeed_value_1,JinsBlinkStrength_value_1,JinsEyeMovement_value_1,JinsEyeMovement_value_2,JinsEyeMovement_value_3,JinsEyeMovement_value_4,JinsGyroscope_value_1,JinsGyroscope_value_2,JinsGyroscope_value_3,PhoneAccelerometer_value_1,PhoneAccelerometer_value_2,PhoneAccelerometer_value_3,PhoneGravity_value_1,PhoneGravity_value_2,PhoneGravity_value_3,PhoneGyroscope_value_1,PhoneGyroscope_value_2,PhoneGyroscope_value_3,PhoneLinearAccelerometer_value_1,PhoneLinearAccelerometer_value_2,PhoneLinearAccelerometer_value_3,PhoneMagnetometer_value_1,PhoneMagnetometer_value_2,PhoneMagnetometer_value_3
timestep (20 ms each),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,-0.193974,-0.739190,-0.208380,-0.110357,-0.350862,-0.043824,-1.056696,-0.470916,-0.328287,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.378683,-0.305279,-1.119933,0.250685,-1.279138,-0.450162,0.268194,-0.949070,-0.537828,-0.045227,-1.699269,-1.744656,0.018173,-2.356718,-0.193671,0.773286,-0.261321,0.075579
1,-0.128987,-0.762604,-0.240084,-0.060584,-0.346152,-0.065252,-1.056696,-0.598604,-0.328287,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.395032,-0.318037,-1.119795,0.207594,-1.319019,-0.630244,0.225061,-0.949555,-0.561715,-0.312748,-1.598247,-1.589829,-0.049190,-2.609550,-0.730282,0.784458,-0.243649,0.061496
2,-0.065476,-0.825961,-0.335590,0.039234,-0.316006,-0.038954,-1.056696,-0.726292,-0.328287,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.411381,-0.330795,-1.119658,0.171417,-1.275568,-0.832666,0.179577,-0.937327,-0.582587,-0.780574,-1.470430,-1.405809,-0.137088,-2.047638,-0.788637,0.817922,-0.214929,0.075563
3,0.037174,-0.941657,-0.437437,0.028083,-0.288686,-0.075966,-1.056696,-0.853981,-0.328287,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.427730,-0.343553,-1.119520,0.027399,-1.187470,-0.720505,0.136715,-0.919398,-0.599820,-1.079177,-1.392051,-1.276143,-0.543829,-1.707668,-0.462143,0.832788,-0.128838,0.041111
4,0.142040,-1.030495,-0.414848,-0.037193,-0.307527,-0.235700,-1.056696,-0.853981,-0.527557,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.443863,-0.364253,-1.118970,-0.069064,-1.145809,-0.631534,0.095780,-0.899574,-0.614974,-1.158403,-1.324037,-1.161514,-0.523160,-1.586216,-0.447007,0.861284,-0.186255,0.097411
5,0.088869,-0.899648,-0.358575,-0.131300,-0.246294,-0.197714,-1.056696,-0.853981,-0.726827,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.459997,-0.384952,-1.118421,-0.038248,-1.077954,-0.738559,0.057090,-0.879033,-0.628291,-1.251958,-1.241523,-1.122325,-0.207908,-1.344888,-0.832780,0.871195,-0.281193,0.095836
6,0.023143,-0.735057,-0.366501,-0.169650,-0.226511,-0.145119,-1.056696,-0.853981,-0.726827,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.467741,-0.406824,-1.117664,-0.024852,-1.041047,-0.968059,0.021887,-0.855982,-0.638599,-1.489428,-1.101361,-1.134380,-0.219015,-1.319393,-1.072650,0.887296,-0.323138,0.103665
7,0.023143,-0.659993,-0.395430,-0.125316,-0.209554,-0.187974,-1.056696,-0.853981,-0.726827,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.475485,-0.428695,-1.116908,-0.074422,-1.016047,-0.886836,-0.009952,-0.829431,-0.645666,-1.694578,-1.000330,-1.131355,-0.386583,-1.000853,-0.635260,0.915805,-0.347412,0.164690
8,0.105854,-0.796349,-0.406922,-0.135924,-0.243468,-0.418810,-1.056696,-1.045513,-0.726827,-0.181746,-0.182212,-0.175064,-0.122574,-0.082490,-0.093299,1.483230,-0.454472,-1.115877,-0.219784,-0.894024,-0.728238,-0.040104,-0.801944,-0.651188,-1.665816,-0.928182,-1.049909,-0.910923,-0.398970,-0.568591,0.936874,-0.307645,0.217903


S8 X_train[0]: shape=(128, 33); label=0 (PressByGrasp)
Source: S8_PressByGrasp_1_extracted / Event_0 / window 0


,android.sensor.accelerometer_value_1,android.sensor.accelerometer_value_2,android.sensor.accelerometer_value_3,android.sensor.gyroscope_value_1,android.sensor.gyroscope_value_2,android.sensor.gyroscope_value_3,JinsAccelerometer_value_1,JinsAccelerometer_value_2,JinsAccelerometer_value_3,JinsBlinkSpeed_value_1,JinsBlinkStrength_value_1,JinsEyeMovement_value_1,JinsEyeMovement_value_2,JinsEyeMovement_value_3,JinsEyeMovement_value_4,JinsGyroscope_value_1,JinsGyroscope_value_2,JinsGyroscope_value_3,PhoneAccelerometer_value_1,PhoneAccelerometer_value_2,PhoneAccelerometer_value_3,PhoneGravity_value_1,PhoneGravity_value_2,PhoneGravity_value_3,PhoneGyroscope_value_1,PhoneGyroscope_value_2,PhoneGyroscope_value_3,PhoneLinearAccelerometer_value_1,PhoneLinearAccelerometer_value_2,PhoneLinearAccelerometer_value_3,PhoneMagnetometer_value_1,PhoneMagnetometer_value_2,PhoneMagnetometer_value_3
timestep (20 ms each),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,-0.531968,-1.031155,-0.561594,0.298488,-0.503932,-0.203238,-1.036704,1.334249,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.652555,1.269178,-1.187978,0.793629,-0.385811,-0.688597,0.817109,-0.397510,-0.710077,0.048021,-0.038823,0.007500,-0.024895,0.018713,0.062572,-0.337714,0.083553,0.245762
1,-0.516841,-0.990434,-0.514840,0.228304,-0.684524,-0.211068,-1.036704,1.334249,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.642510,1.273482,-1.186920,0.789737,-0.385046,-0.691467,0.816620,-0.397670,-0.710657,0.078595,-0.033717,0.026968,-0.034740,0.027122,-0.032223,-0.342103,0.087253,0.238732
2,-0.507764,-0.996357,-0.402767,0.444139,-0.703079,-0.156256,-1.036704,1.334249,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.632465,1.277787,-1.185863,0.783179,-0.380457,-0.704998,0.816114,-0.398081,-0.711488,0.095276,-0.027989,0.041972,-0.091376,-0.003109,0.060562,-0.349904,0.054079,0.261834
3,-0.490822,-1.035598,-0.492838,0.447912,-0.637521,-0.172700,-1.036704,1.334249,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.622421,1.282092,-1.184806,0.770475,-0.392440,-0.697822,0.815990,-0.398671,-0.712160,0.127243,-0.012679,0.098895,-0.078073,0.043039,0.054011,-0.360135,-0.003681,0.278918
4,-0.473274,-0.951194,-0.575346,0.358862,-0.643706,-0.225162,-1.036704,1.186448,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.595112,1.287831,-1.183952,0.769042,-0.380966,-0.688392,0.816538,-0.399843,-0.712802,0.134170,0.007729,0.157364,-0.079397,0.072102,0.029521,-0.358672,-0.039292,0.273891
5,-0.458147,-0.883819,-0.584284,0.132462,-0.500221,-0.261965,-1.036704,1.038647,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.567803,1.293571,-1.183098,0.773345,-0.373828,-0.702744,0.818024,-0.401783,-0.713359,0.114723,0.005180,0.181318,0.012106,0.012217,-0.014430,-0.357204,-0.062616,0.303031
6,-0.456937,-0.840137,-0.578096,0.122651,-0.327049,-0.271361,-1.036704,1.038647,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.556503,1.298593,-1.182854,0.786458,-0.389635,-0.703152,0.819870,-0.404033,-0.713896,0.113349,-0.008212,0.154359,0.072937,-0.038846,0.030408,-0.352815,-0.063828,0.290985
7,-0.445441,-0.951194,-0.571220,0.156611,-0.309732,-0.355145,-1.036704,1.038647,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.545203,1.303615,-1.182610,0.800186,-0.405187,-0.703563,0.821793,-0.406262,-0.714342,0.104989,-0.009491,0.133367,0.151277,-0.018328,-0.008242,-0.354283,-0.017143,0.253824
8,-0.453307,-0.928983,-0.530654,0.479608,-0.273861,-0.186011,-1.036704,1.038647,0.453139,-0.180313,-0.178093,-0.192201,-0.116476,-0.132778,-0.110129,-0.533903,1.308637,-1.182366,0.820468,-0.400343,-0.703563,0.823940,-0.408678,-0.714766,0.121690,-0.005033,0.103424,0.172925,-0.090025,0.046603,-0.353306,-0.001194,0.235749


In [8]:
from datetime import datetime

# Export every training timestep and all sensor features without changing the source arrays.
EXPORT_ROOT = DATA_PREP_DIR / 'all_subject_training_data_exports' / f'run_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)
export_rows = []

for subject, data in SUBJECT_TRAINING.items():
    X = data['X']
    metadata = data['metadata'].reset_index(drop=True)
    sample_count, timestep_count, feature_count = X.shape
    repeated_metadata = metadata.iloc[np.repeat(np.arange(sample_count), timestep_count)].reset_index(drop=True)
    export_frame = repeated_metadata[['subject', 'activity', 'label', 'original_folder', 'split', 'event', 'window_id']].copy()
    export_frame.insert(1, 'sample_index', np.repeat(np.arange(sample_count), timestep_count))
    export_frame.insert(7, 'timestep_index', np.tile(np.arange(timestep_count), sample_count))
    export_frame.insert(8, 'relative_time_ms', export_frame['timestep_index'] * 20)
    values = X.reshape(sample_count * timestep_count, feature_count)
    for feature_index, feature_name in enumerate(data['feature_names']):
        export_frame[feature_name] = values[:, feature_index]

    output_path = EXPORT_ROOT / f'{subject}_all_training_timesteps.csv'
    export_frame.to_csv(output_path, index=False)
    export_rows.append({
        'subject': subject,
        'windows': sample_count,
        'timesteps_per_window': timestep_count,
        'features': feature_count,
        'export_rows': len(export_frame),
        'csv_path': str(output_path),
    })

export_summary = pd.DataFrame(export_rows)
print(f'Complete training-data CSVs saved to: {EXPORT_ROOT.resolve()}')
display(export_summary)
print('Each CSV row is one 20 ms timestep. Columns include subject/activity/label, source folder, Event, window, timestep, and every sensor feature.')

Complete training-data CSVs saved to: D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\DATA_PREP\all_subject_training_data_exports\run_20261003_203214


,subject,windows,timesteps_per_window,features,export_rows,csv_path
0,S1,38,128,33,4864,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
1,S2,25,128,33,3200,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
2,S3,28,128,33,3584,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
3,S4,36,128,33,4608,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
4,S5,37,128,33,4736,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
5,S6,31,128,33,3968,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
6,S7,30,128,33,3840,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...
7,S8,29,128,33,3712,D:\7th sem\ML\DATASETS\Annonmazed_CogActivity\...


Each CSV row is one 20 ms timestep. Columns include subject/activity/label, source folder, Event, window, timestep, and every sensor feature.
